# 02 – Stufe 2: selbst erfinden (GRPO)
Das Modell bekommt nur Fragen und wird belohnt, wenn die Idee **originell und funktionsfähig** ist. Ziel: **200 Schritte in einer Kaggle-Session** (~7 h bei voller Beschleunigung).

**Erster Lauf:** Input = Output von Stufe 1 (`notebook54c7b85763`, enthält `adapter_sft`).
**Fortsetzen:** Input = Output des vorherigen Stufe-2-Laufs (z. B. `creativ-02-grpo`, enthält `checkpoints_grpo`). Das Notebook nimmt automatisch den **neuesten Checkpoint** und macht dort weiter, inklusive Optimizer, Schrittzähler und Datenposition.

**Kaggle:** GPU T4 x2 · Internet an · **Save Version → Save & Run All (Commit)**.
Die ersten ~20 Min laufen drei Schnelltests. Der Lauf stoppt bei 200 Schritten oder nach 10,5 h und speichert.

In [ ]:
import time; START = time.time()   # Uhr für das Zeitbudget läuft ab hier
# === Setup: Repo holen + Abhängigkeiten ===
REPO_URL = "https://github.com/Lommel123/creativ-model.git"
import os, sys, subprocess
if os.path.exists("repo"):
    subprocess.run(["git", "-C", "repo", "pull", "-q"], check=True)
else:
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, "repo"], check=True)
%pip install -q unsloth pyyaml sentence-transformers
sys.path.insert(0, "repo/scripts")
import yaml
CFG = yaml.safe_load(open("repo/config.yaml"))
G, L, RUN = CFG["grpo"], CFG["lora"], CFG["run"]
print(subprocess.run(["git", "-C", "repo", "log", "--oneline", "-1"], capture_output=True, text=True).stdout)

In [ ]:
from unsloth import FastLanguageModel, is_bfloat16_supported
import json, re, glob, shutil, torch
from transformers import TrainerCallback

def chat(tokenizer, messages, add_generation_prompt=False):
    kw = dict(tokenize=False, add_generation_prompt=add_generation_prompt)
    try:
        return tokenizer.apply_chat_template(messages, enable_thinking=False, **kw)  # Qwen3: Thinking-Modus aus
    except TypeError:
        return tokenizer.apply_chat_template(messages, **kw)

def generate(model, tokenizer, question, max_new_tokens=600, temperature=0.8):
    p = chat(tokenizer, [{"role": "user", "content": question}], add_generation_prompt=True)
    ids = tokenizer(p, return_tensors="pt").to("cuda:0")
    out = model.generate(**ids, max_new_tokens=max_new_tokens, temperature=temperature, do_sample=True)
    return tokenizer.decode(out[0][ids.input_ids.shape[1]:], skip_special_tokens=True).strip()

def ensure_max_seq_length(model, n):
    # Absicherung gegen den V1-Fehler: Unsloths schnelle Generierung liest .max_seq_length
    for m in model.modules():
        if type(m).__name__.endswith(("ForCausalLM", "Model")) and not hasattr(m, "max_seq_length"):
            m.max_seq_length = n

class Zeitlimit(TrainerCallback):
    """Stoppt sauber (mit Speichern) vor Kaggles hartem 12-h-Limit."""
    def __init__(self, deadline): self.deadline = deadline
    def on_step_end(self, args, state, control, **kw):
        if time.time() > self.deadline:
            print(f"⏱ Zeitbudget erreicht bei Schritt {state.global_step} – speichere und stoppe.")
            control.should_training_stop = True; control.should_save = True
        return control

DEADLINE = START + CFG["grpo"].get("time_budget_hours", RUN["time_budget_hours"]) * 3600 - 25 * 60   # 25 Min Puffer
def rest(): return f"{(DEADLINE - time.time()) / 3600:.1f} h übrig"
print("GPUs:", torch.cuda.device_count(), "|", rest())

In [ ]:
# === Startpunkt finden: neuester GRPO-Checkpoint (Fortsetzen) oder adapter_sft (Neustart) ===
step_of = lambda p: int(p.rstrip("/").split("-")[-1])
ckpts = [os.path.dirname(p) for p in glob.glob("/kaggle/input/**/checkpoints_grpo/checkpoint-*/trainer_state.json", recursive=True)
         if os.path.exists(os.path.join(os.path.dirname(p), "adapter_model.safetensors"))]
ckpts = sorted(ckpts, key=step_of)
if ckpts:
    src = ckpts[-1]
    RESUME = os.path.join("checkpoints_grpo", os.path.basename(src))
    shutil.copytree(src, RESUME, dirs_exist_ok=True)          # nur den neuesten kopieren (spart Zeit/Platz)
    WEIGHTS = os.path.join(RESUME, "adapter_model.safetensors")
    print(f"▶ FORTSETZEN ab Schritt {step_of(src)}  ({src})")
else:
    RESUME = None
    sft_files = glob.glob("/kaggle/input/**/adapter_sft/adapter_model.safetensors", recursive=True)
    assert sft_files, "Weder checkpoints_grpo noch adapter_sft im Input – rechts 'Add Input → Your Work' den vorherigen Lauf hinzufügen!"
    WEIGHTS = sft_files[0]
    print("▶ NEUSTART von adapter_sft:", WEIGHTS)
if G["max_steps"] and ckpts and step_of(ckpts[-1]) >= G["max_steps"]:
    print(f"⚠ Checkpoint ist schon bei {step_of(ckpts[-1])} ≥ max_steps {G['max_steps']} – in config.yaml max_steps erhöhen, sonst passiert nichts.")

# Basismodell + frisches LoRA (mit Gradient Checkpointing), dann Gewichte hineinladen
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=CFG["base_model"], max_seq_length=CFG["max_seq_length"], load_in_4bit=True)
model = FastLanguageModel.get_peft_model(
    model, r=L["r"], lora_alpha=L["alpha"], lora_dropout=L["dropout"],
    target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"],
    use_gradient_checkpointing="unsloth", random_state=42)

from peft import set_peft_model_state_dict
from safetensors.torch import load_file
sd = load_file(WEIGHTS)
res = set_peft_model_state_dict(model, sd)
missing = [k for k in res.missing_keys if "lora_" in k]
print(f"Gewichte: {len(sd)} Tensoren · fehlende LoRA: {len(missing)} · unerwartet: {len(res.unexpected_keys)}")
assert not missing and not res.unexpected_keys, "Adapter passt nicht zum Basismodell – config.yaml (base_model/lora) prüfen"
ensure_max_seq_length(model, CFG["max_seq_length"])
model.print_trainable_parameters()

In [ ]:
# Schnelltest 1: Generierung des trainierten Modells (genau der Codepfad, der in V1 gecrasht ist)
FastLanguageModel.for_inference(model)
print(generate(model, tokenizer, "Wie hält man Brot länger frisch?", max_new_tokens=60))
FastLanguageModel.for_training(model)
print("✓ Generierung ok |", rest())

In [ ]:
# === Judge (eigener Prozess OHNE Unsloth, GPU 1) + Embeddings ===
from judge_client import Judge
from sentence_transformers import SentenceTransformer
JGPU = 1 if torch.cuda.device_count() > 1 else 0
JUDGE = Judge(G["judge_model"], gpu=JGPU)
embedder = SentenceTransformer(G["embed_model"], device=f"cuda:{JGPU}")
def emb(texts):
    return embedder.encode([f"query: {t}" for t in texts], normalize_embeddings=True, convert_to_tensor=True)

# Schnelltest 2: gute Idee sollte deutlich höher bewertet werden als Floskel
print(JUDGE.validity(["Wie hält man Brot länger frisch?"] * 2,
                     ["Brot in ein Leinentuch wickeln und in einem Brotkasten lagern.", "Einfach kreativ sein."]))
print("✓ Judge ok |", rest())

In [ ]:
# === Belohnungsfunktionen ===
def best_idea(text):
    m = re.search(r"^Beste Idee:\s*(.+)", text, re.M | re.S)
    return (m.group(1) if m else text[-600:]).strip()

def question_of(prompt):
    m = re.search(r"user\n(.*?)<\|im_end\|>", prompt, re.S)
    return m.group(1).strip() if m else prompt[-400:]

def r_format(prompts, completions, **kw):
    out = []
    for c in completions:
        Ls = c.split("\n")
        tools = sum(l.startswith("Werkzeug – ") and "→ Rohidee:" in l for l in Ls)
        s = 0.25 * any(l.startswith("Annahmen:") for l in Ls) + 0.25 * (3 <= tools <= 4)
        s += 0.2 * any(l.startswith("Aussortiert:") for l in Ls) + 0.3 * any(l.startswith("Beste Idee:") for l in Ls)
        out.append(s)
    return out

LOG = []
def r_creative_valid(prompts, completions, obvious, **kw):
    qs = [question_of(p) for p in prompts]
    ideas = [best_idea(c) for c in completions]
    valid = JUDGE.validity(qs, ideas)
    E_i, E_o = emb(ideas), emb([o or q for o, q in zip(obvious, qs)])
    novelty = (1 - (E_i * E_o).sum(-1)).clamp(min=0).mul(4).clamp(max=1).tolist()
    LOG.append({"t_h": round((time.time() - START) / 3600, 2), "valid": sum(valid) / len(valid), "novelty": sum(novelty) / len(novelty)})
    return [v * (0.4 + 0.6 * n) for v, n in zip(valid, novelty)]   # ohne Gültigkeit keine Punkte

def r_diversity(prompts, completions, **kw):
    E = emb([best_idea(c) for c in completions]); out = []
    for i, p in enumerate(prompts):
        others = [j for j, q in enumerate(prompts) if q == p and j != i]
        out.append(0.0 if not others else float((1 - (E[others] @ E[i])).mean().mul(4).clamp(0, 1)))
    return out

LT, LM = G.get("length_target_words", 230), G.get("length_max_words", 265)
def r_length(prompts, completions, **kw):
    # Qwen3: ~1,9 Tokens/Wort → 512 Tokens ≈ 270 Wörter; Strafe muss darunter beginnen, sonst greift sie nie
    return [0.0 if len(c.split()) <= LT else -min(1.0, (len(c.split()) - LT) / (LM - LT)) for c in completions]

W = G["reward_weights"]
REWARDS = [r_format, r_creative_valid, r_diversity, r_length]
WEIGHTS = [W["format"], W["creative_valid"], W["diversity"], W["length"]]

# Schnelltest 3: alle Belohnungen mit echtem Beispiel vs. Floskel
ex = json.loads(open("repo/data/sft.jsonl").readline())
P = chat(tokenizer, [ex["messages"][0]], add_generation_prompt=True)
C = [ex["messages"][1]["content"], "Man könnte einfach kreativ sein und neue Wege gehen."]
for f in REWARDS:
    print(f"{f.__name__:18s}", [round(x, 2) for x in f([P, P], C, obvious=["", ""])])
LOG.clear(); print("✓ Belohnungen ok |", rest())

In [ ]:
from datasets import load_dataset
ds = load_dataset("json", data_files="repo/data/grpo_prompts.jsonl", split="train")
ds = ds.map(lambda r: {"prompt": chat(tokenizer, [{"role": "user", "content": r["prompt"]}], add_generation_prompt=True),
                       "obvious": r["obvious"] or ""}).shuffle(seed=42)
print(len(ds), "Fragen |", rest())

In [ ]:
# === Stufe 2: GRPO – bis max_steps oder Zeitbudget ===
import dataclasses, gc
from trl import GRPOTrainer, GRPOConfig
known = {f.name for f in dataclasses.fields(GRPOConfig)}

def make_trainer(spg):
    kw = dict(
        learning_rate=G["lr"], max_steps=G["max_steps"], save_steps=G["save_steps"], save_total_limit=2,
        per_device_train_batch_size=G["batch_size"], gradient_accumulation_steps=G["grad_accum"],
        steps_per_generation=spg, num_generations=G["num_generations"], temperature=G["temperature"],
        max_prompt_length=G["max_prompt_length"], max_completion_length=G["max_completion_length"],
        mask_truncated_completions=G.get("mask_truncated_completions", False),
        reward_weights=WEIGHTS, optim="adamw_8bit", lr_scheduler_type="constant_with_warmup", warmup_steps=5,
        fp16=not is_bfloat16_supported(), bf16=is_bfloat16_supported(),
        logging_steps=1, output_dir="checkpoints_grpo", report_to="none", seed=42)
    skipped = [k for k in kw if k not in known]
    if skipped: print("Von dieser TRL-Version nicht unterstützt (ignoriert):", skipped)
    t = GRPOTrainer(model=model, processing_class=tokenizer, reward_funcs=REWARDS,
                    args=GRPOConfig(**{k: v for k, v in kw.items() if k in known}),
                    train_dataset=ds, callbacks=[Zeitlimit(DEADLINE)])
    ensure_max_seq_length(t.model, CFG["max_seq_length"])
    return t

def latest():
    c = sorted(glob.glob("checkpoints_grpo/checkpoint-*"), key=step_of)
    return c[-1] if c else None

def is_oom(e):
    return isinstance(e, torch.cuda.OutOfMemoryError) or "out of memory" in str(e).lower()

FastLanguageModel.for_training(model)
SPG = G.get("steps_per_generation") or G["grad_accum"]
print(f"Generierungs-Batch: {G['batch_size'] * SPG} Antworten | Ziel: Schritt {G['max_steps']} | {rest()}")
trainer = make_trainer(SPG)
try:
    trainer.train(resume_from_checkpoint=latest())
except Exception as e:
    if not is_oom(e) or SPG == G["grad_accum"]:
        raise
    print(f"⚠ Speicher reicht nicht für {G['batch_size'] * SPG} Antworten auf einmal – weiter mit {G['batch_size'] * G['grad_accum']} (ab {latest() or 'Start'})")
    del trainer; gc.collect(); torch.cuda.empty_cache()
    SPG = G["grad_accum"]
    trainer = make_trainer(SPG)
    trainer.train(resume_from_checkpoint=latest())

print("Erreicht: Schritt", trainer.state.global_step, "von", G["max_steps"])
model.save_pretrained("adapter_grpo"); tokenizer.save_pretrained("adapter_grpo")
print("Gespeichert: adapter_grpo |", rest())

In [ ]:
import pandas as pd
df = pd.DataFrame(LOG)
if len(df):
    k = max(1, len(df) // 12)
    print("Gültigkeit & Originalität im Verlauf (gleitender Schnitt) – gesund: novelty steigt, valid bleibt stabil")
    print(df.rolling(k).mean().iloc[::k].round(3).to_string(index=False))

FastLanguageModel.for_inference(model)
for f in ("business_holdout.jsonl", "business_evolution.jsonl"):
    q = json.loads(open(f"repo/eval/{f}").readline())["prompt"]
    print("\n" + "=" * 90 + "\nFRAGE (Business, nie trainiert):", q)
    print("\n--- NACH STUFE 2 ---\n", generate(model, tokenizer, q))
JUDGE.close()
print("\nFertig. Output: adapter_grpo (Ergebnis) + checkpoints_grpo (zum Fortsetzen).")